# Format data in digestable way for TTM

In [13]:
%matplotlib inline
import datasets
import datetime

random_state = 42

# Path to your .arrow shard files

folder_path = "/home/joffreyma/TS/Code/JeanZay/chronos-forecasting/data/buildings_900K_chronos_split_ttm/"

output_test_in_domain_path = folder_path + "in_domain/"
output_test_domain_shift_path = folder_path + "domain_shift/"
val_size = 5000
test_domain_shift_size = 14687
prediction_length = 64
context_length = 512

In [14]:
files_eval_test = ["/home/joffreyma/TS/Code/JeanZay/chronos-forecasting/data/buildings_900k/eval/data-00000-of-00001.arrow",
         "/home/joffreyma/TS/Code/JeanZay/chronos-forecasting/data/buildings_900k/test/data-00000-of-00001.arrow"]
ds_eval_test = datasets.load_dataset(
"arrow", data_files={'train': files_eval_test}, split="train", num_proc=2
)

columns = list(ds_eval_test.features.keys())[0:4]
columns

['item_id', 'start', 'freq', 'target']

In [15]:
file_train = ["/home/joffreyma/TS/Code/JeanZay/chronos-forecasting/data/buildings_900k/train/data-00000-of-00001.arrow",]
ds_train = datasets.load_dataset(
    "arrow", data_files={'train': file_train}, split="train", 
)

In [16]:
ds_train = ds_train.select_columns(columns)

In [17]:
ds = datasets.concatenate_datasets([ds_eval_test, ds_train], axis=0)

In [18]:
timestamp = [ds[0]["start"] + datetime.timedelta(hours=h) for h in range(len(ds[0]["target"]))]

In [ ]:
def add_timestamp(example, timestamp):
    # add the timestamp to the example
    return {
        "timestamp": timestamp
    }

In [ ]:
ds = ds.map(add_timestamp, fn_kwargs={"timestamp": timestamp})

In [6]:
del ds_eval_test
del ds_train

In [7]:
temp_domains = ds.train_test_split(test_size=test_domain_shift_size, seed=random_state, shuffle=True)

In [8]:
del ds

In [9]:
ds_in_domain = temp_domains['train']
ds_domain_shift = temp_domains['test']

In [10]:
ds_in_domain.save_to_disk(output_test_in_domain_path, num_shards=1)
ds_domain_shift.save_to_disk(output_test_domain_shift_path, num_shards=1)

Saving the dataset (0/1 shards):   0%|          | 0/60000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14687 [00:00<?, ? examples/s]